In [1]:
import time
import logging
import pandas as pd
from truedata import TD_hist

def fetch_truedata_history(
    username: str,
    password: str,
    ticker_list: list,
    duration: str = '1 Y',
    bar_size: str = 'EOD',
    sleep_time: float = 0.1
) -> tuple[pd.DataFrame, list]:
    """
    Fetches historical data from TrueData for a list of tickers.

    Parameters
    ----------
    username : str
        TrueData username.
    password : str
        TrueData password.
    ticker_list : list
        List of ticker symbols to fetch data for.
    duration : str, optional
        Duration of data (e.g., '1 Y', '25 Y', etc.). Default is '1 Y'.
    bar_size : str, optional
        Bar size for data ('EOD', 'WEEK', etc.). Default is 'EOD'.
    sleep_time : float, optional
        Delay between API calls to avoid throttling. Default is 0.2 seconds.

    Returns
    -------
    final_df : pd.DataFrame
        Combined DataFrame of all tickers' historical data.
    error_list : list
        List of tickers that failed to fetch.
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

    # Initialize connection
    td_hist = TD_hist(username, password)

    df_list = []
    error_list = []

    for ticker in ticker_list:
        try:
            df = td_hist.get_historic_data([ticker], duration=duration, bar_size=bar_size)

            df['Ticker'] = ticker
            df = df.rename(columns={
                'timestamp': 'Date',
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'open': 'Open'
            })

            df_list.append(df)
            logging.info(f"Fetched data for {ticker} ({len(df)} rows).")
            time.sleep(sleep_time)

        except Exception as e:
            logging.error(f"Failed to fetch data for {ticker}: {e}")
            error_list.append(ticker)

    final_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    return final_df, error_list


In [2]:
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import yfinance as yf
from pathlib import Path

import os

def run_momentum_strategy(universe_file: str,
                          start_date: str,
                          end_date: str,
                          top_n: int,
                          output_root: str = "Momentum_Results") -> str:
    """
    Run rolling-window momentum strategy on given stock universe.

    Parameters
    ----------
    universe_file : str
        Path to universe file (.csv or .xlsx) with columns ["Symbol", "ISIN Code"].
    start_date : str
        Start date for backtest (YYYY-MM-DD).
    end_date : str
        End date for backtest (YYYY-MM-DD).
    top_n : int
        Number of top ranked stocks to select each window.
    output_root : str
        Root folder where results will be saved.

    Returns
    -------
    str
        Path to master summary file.
    """


    # ==== 1. Load Universe ====
    if universe_file.endswith(".csv"):
        stock_list = pd.read_csv(universe_file)[["Symbol", "ISIN Code"]]
    else:
        stock_list = pd.read_excel(universe_file)[["Symbol", "ISIN Code"]]
    # username = 'tdwsf695'
    # password = 'ocean@695'
    
    username = 'tdwsf695'
    password = 'ocean@695'
    # username = 'td105'
    # password = 'sharma@105'
    # stock_list["Ticker"] = stock_list["Symbol"] + ".NS"
    stock_list["Ticker"] = stock_list["Symbol"] 
    symbol_list = stock_list["Symbol"].tolist()
    universe_name = Path(universe_file).stem
    output_dir = os.path.join(output_root, f"{universe_name}_{top_n}_stocks_results")
    os.makedirs(output_dir, exist_ok=True)

    # ==== 2. Download Data ====
    total_start = pd.to_datetime(start_date)
    total_end = pd.to_datetime(end_date)

    print(f"\n📥 Downloading price data for {len(symbol_list)} symbols...")
    # data = yf.download(symbol_list, start=total_start.strftime('%Y-%m-%d'),
    #                    end=total_end.strftime('%Y-%m-%d'), progress=True)
    # prices_all = data["Close"]
    # prices_all.index = pd.to_datetime(prices_all.index)
    # print("✅ Download complete.")

    data, errors = fetch_truedata_history(username, password, symbol_list, duration='10 Y', bar_size='EOD')
    data = data[['Date', 'Close', 'Ticker']]
    data.drop_duplicates(subset=['Date', 'Ticker'], inplace=True)
    print(data)
    print("Failed tickers:", errors)
    prices = data.pivot(index="Date", columns="Ticker", values="Close")  
    # optional: sort by date
    prices_all = prices.sort_index()



    # ==== 3. Create Rolling Windows ====
    windows = []
    current_start = total_start
    while True:
        current_end = current_start + relativedelta(months=6)
        if current_end > total_end:
            break
        window_prices = prices_all.loc[(prices_all.index >= current_start) & (prices_all.index < current_end)].copy()
        if not window_prices.empty:
            windows.append((current_start, current_end, window_prices))
        current_start += relativedelta(months=1)

    print(f"📊 Created {len(windows)} rolling windows.")

    # ==== 4. Process Each Window ====
    for start, end, prices in windows:
        file_suffix = f"{start.strftime('%Y%m%d')}_{end.strftime('%Y%m%d')}"
        print(f"\n🔍 Processing window: {start.date()} → {end.date()}")

        prices.dropna(axis=1, how='all', inplace=True)
        if prices.empty:
            print("⚠️ All price data missing. Skipping window.")
            continue

        # Monthly momentum
        monthclose = prices.groupby(prices.index.strftime('%Y-%m')).tail(1)
        monthstart = prices.groupby(prices.index.strftime('%Y-%m')).head(1)
        monthstart.index = monthclose.index
        monchange = (monthclose - monthstart) / monthstart
        MOM = (monchange + 1).product() - 1
        mom = MOM * 100

        # Daily returns
        daily_ret = prices.pct_change(fill_method=None)
        positivechange = (daily_ret[daily_ret > 0].count() / daily_ret.count()) * 100
        negativechange = (daily_ret[daily_ret < 0].count() / daily_ret.count()) * 100

        result = pd.concat([positivechange, negativechange, mom], axis=1, join='inner')
        result.columns = ["Positive", "Negative", "Momentum"]
        result = result.reset_index().rename(columns={'index': 'Ticker'})

        # Merge ISIN
        result = pd.merge(result, stock_list[["Ticker", "ISIN Code"]], on="Ticker", how="left")

        # Ranking
        df = result.copy()
        df["Rank_Mom"] = df["Momentum"].rank(method='min', ascending=False)
        print('rank df:', df)
        df['FIP'] = df.apply(lambda row: row['Negative'] - row['Positive'] if row['Momentum'] > 0 else np.nan, axis=1)
        # df['FIP'] = df.apply(lambda row: row['Negative'] - row['Positive'])

        df.dropna(inplace=True)
        df["FIP_rank"] = df["FIP"].rank(method="first", ascending=True)
        df["Combined_Rank"] = df["Rank_Mom"] + df["FIP_rank"]
        if end.strftime('%Y-%m-%d') == '2026-01-01':
            df = df[~(df['Ticker'].isin(['MARUTI', 'PTCIL']))]

        df = df.sort_values(by="Combined_Rank", ascending=True)

        # ==== Filter: Remove stocks with Current Market Price > ₹7500 ====
        # Filter is applied BEFORE head(top_n) so final output always has top_n stocks
        last_close = prices.iloc[-1]  # Series: Ticker -> last close price
        df["CMP"] = df["Ticker"].map(last_close)
        before_filter = len(df)
        df = df[df["CMP"] <= 7500].copy()
        filtered_out = before_filter - len(df)
        if filtered_out > 0:
            print(f"🚫 Filtered out {filtered_out} stock(s) with CMP > ₹7,500")

        df = df.head(top_n)
        df["Real_Rank"] = range(1, len(df) + 1)
        df["End_Date"] = end.strftime('%Y-%m-%d')


        # Save each window
        output_file = os.path.join(output_dir, f"momentum_{file_suffix}.xlsx")
        df.to_excel(output_file, index=False)
        print(f"✅ Saved results to: {output_file}")

    # ==== 5. Master File ====
    print("\n📂 Creating master summary file...")
    master_data = []
    for file in os.listdir(output_dir):
        if file.startswith("momentum_") and file.endswith(".xlsx"):
            df = pd.read_excel(os.path.join(output_dir, file))
            selected_df = df[["End_Date", "ISIN Code", "Ticker", 'Real_Rank']].copy()
            master_data.append(selected_df)

    if master_data:
        master_df = pd.concat(master_data, ignore_index=True)
        master_file_path = os.path.join(output_dir, "master_momentum_summary.xlsx")
        master_df.to_excel(master_file_path, index=False)
        print(f"✅ Master file saved to: {master_file_path}")
        return master_file_path
    else:
        print("⚠️ No window files found, master not created.")
        return None


In [3]:


# Momentum/Automating Momentum/Universe/Nifty_500_2025_Apr.csv
# "C:\Users\Admin\Momentum\Automating Momentum\Universe\Nifty_500_2025_Apr.csv"
master_file = run_momentum_strategy(
    universe_file=r"C:\Users\LENOVO\Desktop\Ocean Finvest\MOMENTUM_DB_2 copy\Universe\Nifty_500_2025_Apr.csv",
    start_date="2022-06-01",
    end_date="2026-04-01",
    top_n=20,
    output_root="Stocks"
)
print("Master summary located at:", master_file)


📥 Downloading price data for 503 symbols...


(2026-04-29 15:21:57,309) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:24632 Thread:5680)
2026-04-29 15:21:57,309 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-04-29 15:21:57,707 - INFO - Fetched data for 360ONE (1640 rows).
2026-04-29 15:21:58,276 - INFO - Fetched data for 3MINDIA (2478 rows).
2026-04-29 15:21:58,930 - INFO - Fetched data for ABB (2478 rows).
2026-04-29 15:21:59,567 - INFO - Fetched data for ACC (2478 rows).
2026-04-29 15:22:00,102 - INFO - Fetched data for ACMESOLAR (360 rows).
2026-04-29 15:22:00,692 - INFO - Fetched data for AIAENG (2478 rows).
2026-04-29 15:22:01,305 - INFO - Fetched data for APLAPOLLO (2478 rows).
2026-04-29 15:22:01,821 - INFO - Fetched data for AUBANK (2182 rows).
2026-04-29 15:22:02,372 - INFO - Fetched data for AWL (1046 rows).
2026-04-29 15:22:02,823 - INFO - Fetched data for AADHARHFC (485 rows).
2026-04-29 15:22:03,361 - INFO - Fetched data for AARTIIND (2478 rows).
2026-0

              Date    Close  Ticker
0       2019-09-19   317.65  360ONE
1       2019-09-20   333.50  360ONE
2       2019-09-23   350.15  360ONE
3       2019-09-24   367.65  360ONE
4       2019-09-25   351.25  360ONE
...            ...      ...     ...
1038445 2026-04-23  1484.50  ECLERX
1038446 2026-04-24  1468.00  ECLERX
1038447 2026-04-27  1479.30  ECLERX
1038448 2026-04-28  1449.10  ECLERX
1038449 2026-04-29  1412.00  ECLERX

[1038450 rows x 3 columns]
Failed tickers: ['DUMMYABFRL', 'DUMMYSIEMS', 'DUMMYRAYMN', 'SWANENERGY', 'TATAMOTORS']
📊 Created 41 rolling windows.

🔍 Processing window: 2022-06-01 → 2022-12-01
rank df:          Ticker   Positive   Negative   Momentum     ISIN Code  Rank_Mom
0        360ONE  50.806452  48.387097   8.130084  INE466L01038     249.0
1       3MINDIA  47.580645  52.419355  13.148699  INE470A01017     206.0
2      AARTIIND  55.645161  44.354839  -2.403133  INE769A01020     341.0
3         AAVAS  45.161290  54.838710  -8.645213  INE216P01012     380.0
4  

In [4]:
# ----------------xxxxxxxxxxxxxxx---------------------------------------